# Langfuse Tracing 기초

셀프호스팅한 Langfuse에 트레이스를 보내는 기본 패턴을 실습한다.

**사전 조건**
- `docker compose up -d` 로 Langfuse 스택 기동 완료 (http://localhost:3000)
- 상위 디렉토리의 `.env` 에 `LANGFUSE_PUBLIC_KEY` / `LANGFUSE_SECRET_KEY` / `LANGFUSE_HOST` 설정 완료

**다루는 내용**
1. SDK 연결 확인
2. `@observe` 데코레이터 — 함수 단위 자동 트레이싱
3. span / generation 수동 생성
4. LangChain 콜백 연동 (Ollama)

In [12]:
from dotenv import load_dotenv

# 상위 디렉토리의 .env 에서 LANGFUSE_* 환경변수 로드
load_dotenv("../.env")

True

In [13]:
import langfuse
print(langfuse.__version__)
print(langfuse.__file__)   # 어느 경로의 패키지가 import 됐는지도 확인

4.13.0
C:\Users\adot\.local\share\mamba\envs\langfuse-lab\Lib\site-packages\langfuse\__init__.py


In [14]:
from langfuse import get_client

langfuse = get_client()

# True 가 나오면 연결 성공
langfuse.auth_check()

True

## 1. `@observe` 데코레이터

함수에 붙이면 호출마다 트레이스가 생성된다. 중첩 호출은 자동으로 부모-자식 span 으로 연결된다.

실행 후 http://localhost:3000 → Tracing → Traces 에서 확인.

In [15]:
from langfuse import observe


@observe()
def preprocess(text: str) -> str:
    return text.strip().lower()


@observe()
def pipeline(user_input: str) -> str:
    cleaned = preprocess(user_input)  # 자식 span 으로 기록됨
    return f"processed: {cleaned}"


pipeline("  Hello Langfuse!  ")

'processed: hello langfuse!'

## 2. span / generation 수동 생성

데코레이터 없이 트레이스 구조를 직접 제어하는 low-level API.

SDK v4부터는 `start_as_current_observation(as_type=...)` 하나로 통합되었다 (v3의 `start_as_current_span` / `start_as_current_generation`은 제거됨).

- **`as_type="span"`** (기본값): 일반 작업 단위 (전처리, 검색, 후처리 등)
- **`as_type="generation"`**: LLM 호출 전용 — model, usage(토큰), input/output 을 기록하면 대시보드에서 비용이 집계된다
- 그 외 `tool`, `agent`, `chain`, `retriever`, `embedding` 등 — 에이전트 트레이스 시각화용 타입

In [16]:
with langfuse.start_as_current_observation(
    name="rag-pipeline", as_type="span", input={"query": "vLLM이 뭐야?"}
) as root:
    # 검색 단계
    with langfuse.start_as_current_observation(name="retrieval", as_type="span") as retrieval:
        docs = ["vLLM은 LLM 추론·서빙 엔진이다."]
        retrieval.update(output={"docs": docs})

    # LLM 호출 단계 (여기서는 모의 응답)
    with langfuse.start_as_current_observation(
        name="answer",
        as_type="generation",
        model="mock-model",
        input=[{"role": "user", "content": "vLLM이 뭐야?"}],
    ) as gen:
        answer = "vLLM은 PagedAttention 기반의 고성능 LLM 서빙 엔진입니다."
        gen.update(
            output=answer,
            usage_details={"input": 12, "output": 25},  # 토큰 수 기록 → 비용 집계
        )

    root.update(output={"answer": answer})

## 3. LangChain 콜백 연동 (Ollama)

`CallbackHandler` 를 넘기면 체인 내부의 모든 단계(프롬프트, LLM 호출, 파서)가 자동으로 트레이스된다.

> Ollama 서버가 필요하다. `base_url` 을 환경에 맞게 수정할 것 (로컬: `http://localhost:11434`, GPU 서버: `http://<server-ip>:11434`).

In [17]:
from langchain_ollama import ChatOllama
from langfuse.langchain import CallbackHandler

handler = CallbackHandler()

llm = ChatOllama(
    model="gemma4:e4b",  # ollama list 로 확인한 모델명으로 변경
    base_url="http://localhost:11434",
)

response = llm.invoke(
    "LLMOps가 뭔지 한 문장으로 설명해줘.",
    config={"callbacks": [handler]},
)
print(response.content)

LLMOps는 **거대 언어 모델(LLM)을 개발 단계에만 머무르지 않고, 실제 서비스 환경에 안정적이고 지속 가능하게 배포하고 운영하며 성능과 효율성을 관리하는 전체 프로세스**를 의미합니다.


## 4. flush

SDK는 트레이스를 **백그라운드에서 배치 전송**한다. 스크립트/노트북이 끝나기 전에 전송을 보장하려면 flush 를 호출한다.

In [ ]:
langfuse.flush()